# W9｜打開黑盒子 – SHAP 模型解釋
**智慧醫療與機器學習基礎**　臺北醫學大學 智慧醫療學士學位學程　授課教師：王三源　課程助教：古珉瑄（醫資所，m610115009@tmu.edu.tw）

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/wang3yuan/FIHML/blob/main/notebooks/W09_shap.ipynb)

### 今天的情境
W7 你們訓練了隨機森林，幫機構篩出「可能失智」的住民。機構主任看了結果，又問了兩個問題：
> 「這位阿嬤為什麼被判成高風險？」
> 「整體來說，模型到底看重哪些資料？我們該多注意什麼？」

模型說「高風險」卻講不出理由，照顧人員很難放心使用。今天用 **SHAP** 打開模型，同時解開 W7 留下的兩個謎：
1. 為什麼隨機森林說**身高**很重要？（身高和失智在這份資料裡完全無關）
2. 為什麼 W6 看到睡眠和失智是 **U 型**，羅吉斯回歸卻抓不到？

> ⚠️ 本資料為**教學用模擬資料**，不對應任何真實個人。資料欄位說明見 [長照常用評估量表說明（PDF）](https://github.com/wang3yuan/FIHML/blob/main/docs/W05_長照評估量表說明.pdf)。
> ⚠️ 模型是**篩檢輔助**，不是診斷；SHAP 解釋的是「模型怎麼想」，**不是「失智的原因」**（第 3 堂會說明）。

### 執行環境
本教材已在 **Python 3.12＋scikit-learn 1.6.1／pandas 2.2.3** 與 **scikit-learn 1.9.1／pandas 3.0.6** 兩組版本測試通過，兩組都搭配 **shap 0.52.0**。執行下面第一格會印出你目前的版本。
不同版本的 shap 指令寫法可能不同，**查手冊或問 Gemini 時請說明你用的是 shap 0.52.0**。

### 使用方式
1. 「檔案 → 在雲端硬碟中儲存複本」，之後都在自己的複本上作業。
2. **由上往下依序執行**（每格按 ▶，或 Shift＋Enter）；看到 **▶ 實作** 的地方，請在 `# TODO` 處寫程式，或在「✏️ 你的答案」後面作答。
3. 卡住的時候，可以問 Colab 的 Gemini（使用指引見下一格），或打開實作下面的「💡 卡住了再打開」。

### 🤖 在 Colab 用 Gemini 當助教
**怎麼開啟**
- 點 Colab 介面上的 **Gemini 圖示（✦）** 開啟對話視窗，用中文問就可以。
- 程式出錯時，錯誤訊息下方會出現 **「解釋錯誤」** 按鈕，按下去會說明原因並建議怎麼改。

**好的問法**（可以直接複製）
- 「我用 shap **0.52.0**，`shap.plots.waterfall` 怎麼讀？用中文舉例」
- 「逐行解釋這段程式碼在做什麼」（把你看不懂的那幾行貼上去）
- 「我出現這個錯誤：（貼上錯誤訊息最後一行），可能是什麼原因？」
- 「SHAP 值和羅吉斯回歸的係數有什麼不同？」

**不好的問法**
- 「幫我寫出實作 9-2 的答案」：實作是練習自己照範例改，直接要答案就學不到了。

**使用規則**（詳見課程〈生成式 AI 使用指引〉）
1. **一定要查證**：Gemini 可能用舊版的 shap 寫法，貼上去會出錯。數字以你自己跑出來的為準。
2. **作業用了要揭露**：寫在作業最後的「GenAI 使用說明」那一格（用了什麼、怎麼問、怎麼查證）。
3. **不貼個人資料**：這門課的模擬資料是公開的，可以；實習或工作中接觸到的個案資料，不可以。

### 💡 寫程式不熟也沒關係：三個小技巧
1. **照著範例改**：每個實作都有一段範例可以參考，通常只要改「名稱」或「數字」，不用從零開始寫。
2. **每次只改一個地方，馬上執行**：出錯了，就知道是剛剛改的那裡。
3. **錯誤訊息只看最後一行**：最後一行會告訴你錯在哪（例如 `NameError: name 'sv1' is not defined` ＝ 變數 `sv1` 還沒建立，先回去執行前面的格子）。

| 常見錯誤 | 原因 | 怎麼解決 |
|---|---|---|
| `NameError: name 'xxx' is not defined` | 前面的格子還沒執行（或重新連線後變數不見了） | 「執行階段 → 全部執行」，或從上往下重跑 |
| `ModuleNotFoundError: No module named 'shap'` | 還沒安裝 shap | 執行下面第一格 |
| `The waterfall plot can currently only plot a single explanation` | 一次給了很多位住民 | 只給一位：`sv1[0]`（方括號裡放第幾位） |
| `KeyError: 'xxx'` 或 `ValueError: ... not in list` | 欄位名稱打錯 | 用 `X.columns` 看正確的名稱 |

In [ ]:
# 第一次執行會自動安裝 shap（約 1 分鐘，Colab 重新連線後要再裝一次）
try:
    import shap
except ModuleNotFoundError:
    %pip install -q shap==0.52.0
    import shap

import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sklearn

print("Python 版本：", sys.version.split()[0])
print("scikit-learn 版本：", sklearn.__version__)
print("pandas 版本：", pd.__version__)
print("shap 版本：", shap.__version__)

## 0. 設定中文字型（每次開啟 notebook 都要執行一次）
和 W6、W7 一樣：下載思源黑體（Noto Sans TC，開源授權 OFL），讓圖上的中文正常顯示。

In [ ]:
import urllib.request
import matplotlib.font_manager as fm

FONT_URL = "https://raw.githubusercontent.com/wang3yuan/FIHML/main/fonts/"
for name in ["NotoSansTC-Regular.ttf", "NotoSansTC-Bold.ttf"]:
    urllib.request.urlretrieve(FONT_URL + name, name)
    fm.fontManager.addfont(name)

plt.rcParams["font.family"] = "Noto Sans TC"
plt.rcParams["axes.unicode_minus"] = False   # 讓負號正常顯示
plt.rcParams["figure.dpi"] = 110

C_NO, C_YES = "#00897B", "#D06F1A"          # 非失智、失智（和 W6、W7 相同）
print("設定完成")

## 回顧：W7 的資料和模型（直接執行，不用改）
下面幾格是 W7 做過的事：讀資料、整理特徵、切分訓練集與測試集、訓練隨機森林與羅吉斯回歸。

> ⚠️ 特徵只能用「預測當下」拿得到的資訊：確診之後才有的資料（CDR 分期、失智症用藥）不能放進模型。這份資料已經事先拿掉了。

In [ ]:
BASE = "https://raw.githubusercontent.com/wang3yuan/FIHML/main/data/ltc/"
df = pd.read_csv(BASE + "ltc_dementia_clean.csv")

data = df.dropna()                                    # 把有缺值的住民拿掉
X = data.drop(columns=["resident_id", "dementia"])    # 特徵：住民編號不是特徵，答案也不能放進來
X = pd.get_dummies(X, drop_first=True, dtype=int)     # 文字欄位（機構、性別、血型）轉成 0／1
y = data["dementia"]                                  # 標籤：1 = 失智、0 = 非失智

print("住民數：", len(data), "　特徵數：", X.shape[1])
X.head()

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=1151)

forest = RandomForestClassifier(random_state=1151).fit(X_train, y_train)
logit = make_pipeline(StandardScaler(), LogisticRegression()).fit(X_train, y_train)

print("隨機森林 測試集正確率：", round(forest.score(X_test, y_test), 3))
print("羅吉斯回歸 測試集正確率：", round(logit.score(X_test, y_test), 3))

---
# 第 1 堂｜為什麼要解釋模型？認識 SHAP
## 1.1 W7 的特徵重要性不夠用
W7 用 `forest.feature_importances_` 看哪些特徵重要。但它有三個問題：
1. **看不出方向**：只知道「重要」，不知道是 SPMSQ 錯越多越危險，還是越少越危險。
2. **只有整體，沒有個人**：回答不了「這位阿嬤為什麼高風險？」
3. **會高估數值種類很多的特徵**（W7 實作 7-2(d) 的 Warning）：身高有很多不同的數值，隨便切也能分開幾個人。

回頭看一次 W7 的結果：

In [ ]:
imp = pd.Series(forest.feature_importances_, index=X.columns).sort_values(ascending=False)
print("身高（height_cm）的重要性排第", list(imp.index).index("height_cm") + 1, "名，比 23 個特徵裡的大多數都高")
imp.head(8).round(3)

## 1.2 SHAP 是什麼？
想像三位同學合作做報告，拿到一個總成績。要怎麼公平地說「每個人貢獻了多少」？

**SHAP**（SHapley Additive exPlanations，源自賽局理論的 Shapley 值）的做法：把「每個特徵」當成團隊的成員，一位一位加進來，看**預測的機率改變了多少**，再把所有可能的加入順序平均起來。結果是：

> **這位住民的預測機率 ＝ 基準值（所有人的平均）＋ 每個特徵的 SHAP 值加總**

| | 意思 | 例子 |
|---|---|---|
| **基準值** | 還沒看任何資料時，模型的平均預測 | 平均失智機率 0.44 |
| **SHAP 值 > 0** | 這個特徵把預測**往失智推高** | SPMSQ 錯 9 題：+0.25 |
| **SHAP 值 < 0** | 這個特徵把預測**往非失智拉低** | 巴氏量表 95 分：−0.08 |
| **SHAP 值 ≈ 0** | 這個特徵對這位住民沒什麼影響 | 血型：0.00 |

SHAP 值的**單位和模型輸出相同**：我們的隨機森林輸出「失智機率」，所以 SHAP 值就是「這個特徵讓失智機率增加（或減少）了多少」。

## 1.3 算出 SHAP 值
shap 套件的用法也是三個步驟：**建立解釋器 → 算 SHAP 值 → 畫圖**。
- `shap.TreeExplainer(模型)`：專門解釋樹模型（決策樹、隨機森林），又快又精確
- `explainer(X_test)`：對測試集每一位住民、每一個特徵算出 SHAP 值

In [ ]:
explainer = shap.TreeExplainer(forest)     # 1. 建立解釋器（要給它訓練好的模型）
sv = explainer(X_test)                     # 2. 算測試集每位住民、每個特徵的 SHAP 值

print("形狀：", sv.shape)

形狀 `(195, 23, 2)` 的意思：**195 位住民 × 23 個特徵 × 2 個類別**。隨機森林同時解釋「非失智」和「失智」兩個類別；我們只關心「失智」，也就是最後一個數字 `1`：

`sv[:, :, 1]`：所有住民（第一個冒號）、所有特徵（第二個冒號）、第 1 類（失智）。我們把它存成 `sv1`，**接下來的圖都用 `sv1`**。

In [ ]:
sv1 = sv[:, :, 1]
print("形狀：", sv1.shape)
print("基準值（平均失智機率）：", round(float(sv1.base_values[0]), 3))

### 驗證：基準值＋SHAP 值加總 ＝ 預測機率
拿第 0 位住民確認（`.values` 是 SHAP 值，`.base_values` 是基準值）：

In [ ]:
i = 0
base = sv1.base_values[i]
shap_sum = sv1.values[i].sum()
proba = forest.predict_proba(X_test)[i, 1]

print(f"基準值 {base:.3f} ＋ SHAP 值加總 {shap_sum:+.3f} ＝ {base + shap_sum:.3f}")
print(f"模型預測的失智機率：{proba:.3f}")

兩個數字一樣：SHAP 沒有另外猜，只是把模型的預測**拆開來**，看每個特徵各貢獻多少。

## 1.4 整體來看：哪些特徵最重要？
把每個特徵的 SHAP 值取**絕對值**再對所有住民**平均**（`mean(|SHAP|)`），就知道這個特徵平均讓預測改變多少。

In [ ]:
shap.plots.bar(sv1, max_display=10, show=False)
plt.title("平均 |SHAP 值|：平均讓失智機率改變多少")
plt.xlabel("平均 |SHAP 值|（失智機率）")
plt.show()

跟 W7 的特徵重要性比一比：SPMSQ 還是第一名，但**身高掉到後面去了**。把兩種排名放在一起：

In [ ]:
shap_imp = pd.Series(np.abs(sv1.values).mean(axis=0), index=X.columns).sort_values(ascending=False)
rank = pd.DataFrame({"SHAP 排名": shap_imp.rank(ascending=False).astype(int),
                     "W7 特徵重要性排名": imp.rank(ascending=False).astype(int)})
rank.loc[["spmsq_errors", "sleep_hours", "height_cm", "weight_kg", "blood_type_O"]]

- **身高**：W7 重要性排第 6，SHAP 排第 15：SHAP 不會被「數值種類很多」騙到。
- 特徵重要性（不純度下降）衡量「模型**切**了幾次」，SHAP 衡量「**預測被推了多少**」。
- 但**平均絕對值**仍然看不出方向。下一張圖補上這個資訊。

### 蜂群圖（beeswarm）：方向、大小一次看
- **每一個點是一位住民**
- **橫軸**：SHAP 值。右邊（正）＝把這位住民推向失智；左邊（負）＝拉向非失智
- **顏色**：特徵值的高低。**紅色＝這個特徵的值高，藍色＝值低**
- **縱軸**：由上而下依重要性排序

In [ ]:
shap.plots.beeswarm(sv1, max_display=10, show=False)
plt.title("每位住民、每個特徵對失智機率的影響")
plt.show()

怎麼讀：看最上面的 `spmsq_errors`（SPMSQ 錯誤題數）
- **紅點**（錯很多題）都在**右邊**：錯越多題，失智機率被推得越高
- **藍點**（錯很少題）都在**左邊**：錯越少題，失智機率被拉得越低
- 和臨床經驗一致：SPMSQ 錯誤越多，認知功能越差。

再看 `adl_barthel`（巴氏量表）：紅點（分數高、日常生活獨立）在**左邊**，藍點（分數低）在右邊：生活功能越好，失智機率越低。

> 💡 SHAP 同時告訴你「**哪個特徵重要**」和「**往哪個方向影響**」，這是特徵重要性做不到的。

---
## ▶ 實作 9-1（約 10 分鐘）讀整體解釋圖
> 🆘 **卡住了怎麼辦**：① 看錯誤訊息的最後一行 → ② 問 Colab 的 Gemini（按「解釋錯誤」，或請它用中文解釋）→ ③ 還是卡住，再打開下面的「💡 卡住了再打開」。看懂之後，自己再打一次。

**(a)** 只畫前 **5** 名特徵的蜂群圖（提示：`max_display=5`，其他照上面的範例）。

In [ ]:
# 範例：shap.plots.beeswarm(sv1, max_display=10, show=False)
# TODO：改成只畫前 5 名，並加上自己的標題 plt.title("...")
plt.show()


In [ ]:
#@title 💡 卡住了再打開：參考解答 { display-mode: "form" }
shap.plots.beeswarm(sv1, max_display=5, show=False)
plt.title("前 5 名特徵")
plt.show()

**(b)** 看 `falls_past_year`（過去一年跌倒次數）：紅點（跌倒多）在左邊還是右邊？用一句話解讀。

> ✏️ 你的答案：


<details><summary>💡 卡住了再打開：參考答案</summary>

**參考答案：**紅點（跌倒次數多）在**右邊**，SHAP 值為正：過去一年跌倒越多次，模型越傾向預測失智。藍點（沒跌倒）都集中在左邊稍微偏負的地方。

</details>

**(c)** 查出 `sleep_hours`（睡眠時數）的平均 |SHAP| 排第幾名？（提示：用上面的 `shap_imp`，`list(shap_imp.index).index("sleep_hours") + 1` 是排名，要記得 +1）

In [ ]:
# 範例：身高的排名
print("身高的 SHAP 排名：", list(shap_imp.index).index("height_cm") + 1)
# TODO：改成睡眠時數 sleep_hours 的排名


In [ ]:
#@title 💡 卡住了再打開：參考解答 { display-mode: "form" }
print("睡眠時數的 SHAP 排名：", list(shap_imp.index).index("sleep_hours") + 1)
# 排第 5：W7 的特徵重要性排第 3，SHAP 排第 5，順序略有不同，都在前段

---
# 第 2 堂｜解釋一位住民，解開睡眠之謎
## 2.1 瀑布圖（waterfall）：這位住民為什麼被判高風險？
回到主任的問題：「這位阿嬤為什麼被判成高風險？」先找出測試集裡，模型預測失智機率**最高**的住民：

In [ ]:
p = forest.predict_proba(X_test)[:, 1]      # 每位住民的失智機率
i = int(np.argmax(p))                       # 機率最高的是第幾位
print("第", i, "位住民，模型預測失智機率", round(p[i], 2), "，實際診斷（1 = 失智）：", y_test.iloc[i])
X_test.iloc[[i]].T.loc[["age", "spmsq_errors", "adl_barthel", "iadl", "falls_past_year", "sleep_hours", "hearing_loss"]]

In [ ]:
shap.plots.waterfall(sv1[i], max_display=8, show=False)
plt.show()

### 瀑布圖怎麼讀（由下往上）
1. **最下面的 `E[f(X)]`**：基準值（0.44），所有人的平均預測。
2. **每一條橫條**：一個特徵把預測推了多少。**紅色往右＝推高失智機率，藍色往左＝拉低**。左邊灰字是這位住民的**特徵值**（例如 `spmsq_errors = 8`）。
3. **最上面的 `f(x)`**：最後的預測機率，也就是 0.44 加上所有橫條的總和。

> 💡 注意橫軸的單位是**機率**：條上的 `+0.19` ＝ 這個特徵讓失智機率增加 19 個百分點。

### 用程式把「前 3 名推高、前 3 名拉低」列出來
瀑布圖看得清楚，但報告裡常需要用文字說明。SHAP 值也是一般的數字，可以排序：

In [ ]:
one = pd.Series(sv1.values[i], index=X.columns)       # 這位住民每個特徵的 SHAP 值
print("推高失智機率最多的 3 個特徵：")
print(one.sort_values(ascending=False).head(3).round(3))
print("\n拉低失智機率最多的 3 個特徵：")
print(one.sort_values().head(3).round(3))

### 模型漏掉的住民：為什麼？
這個模型的敏感度只有約 0.7，也就是會**漏掉**一些真正失智的人。看看漏掉的住民，模型是怎麼想的：

In [ ]:
missed = np.where((y_test.values == 1) & (p < 0.5))[0]   # 真的失智、但模型預測機率 < 0.5
print("測試集中被漏掉的失智住民：", len(missed), "位")
j = int(missed[0])
print("例如第", j, "位：模型預測失智機率只有", round(p[j], 2))

In [ ]:
shap.plots.waterfall(sv1[j], max_display=8, show=False)
plt.show()

這位住民真的有失智，但模型給的機率只有 0.35。從圖可以看出：SPMSQ **一題都沒答錯**，把預測往下拉了 0.16，蓋過了「睡 9.1 小時」（+0.12）和年紀 90 歲（+0.05）的推高，模型因此被「騙」了。**解釋可以幫我們找出模型的盲點**，例如：這類住民的資料有沒有缺漏？需不需要補充更多量表？

---
## ▶ 實作 9-2（約 12 分鐘）解釋兩位住民
> 🆘 **卡住了怎麼辦**：① 看錯誤訊息的最後一行 → ② 問 Colab 的 Gemini（按「解釋錯誤」，或請它用中文解釋）→ ③ 還是卡住，再打開下面的「💡 卡住了再打開」。看懂之後，自己再打一次。

**(a)** 第 1 位：模型預測**失智機率最高**的那位（上面的 `i`）。從瀑布圖或排序，說出**推高最多的 3 個特徵**與它們的特徵值。

> ✏️ 你的答案：


<details><summary>💡 卡住了再打開：參考答案</summary>

**參考答案：**推高最多的是 `spmsq_errors`（+0.22，錯 6 題）、`sleep_hours`（+0.10，睡 9.8 小時，太多）、`iadl`（+0.07，只有 1 分）：認知測驗錯很多、睡太多、工具性日常生活功能差，都把失智機率往上推。

</details>

**(b)** 第 2 位：**不是失智、但模型預測失智機率最高**的住民（誤報）。在 `X_test` 的位置（第幾位）怎麼找？程式範例如下，請補完後畫出瀑布圖。

In [ ]:
false_alarm = np.where((y_test.values == 0) & (p >= 0.5))[0]    # 沒有失智、但模型預測機率 >= 0.5
# TODO：false_alarm 是一串位置；從中挑出 p 最高的那位，存成 k（參考上面「機率最高」的做法）
# TODO：畫出第 k 位的瀑布圖：shap.plots.waterfall(sv1[k], max_display=8, show=False)
plt.show()


In [ ]:
#@title 💡 卡住了再打開：參考解答 { display-mode: "form" }
false_alarm = np.where((y_test.values == 0) & (p >= 0.5))[0]    # 沒有失智、但模型預測機率 >= 0.5
k = int(false_alarm[np.argmax(p[false_alarm])])                 # 其中機率最高的那位
print("第", k, "位住民，模型預測失智機率", round(p[k], 2), "，實際診斷（1 = 失智）：", y_test.iloc[k])

shap.plots.waterfall(sv1[k], max_display=8, show=False)
plt.show()

**(c)** 這位非失智的住民，是哪些特徵讓模型誤報？你覺得需要照顧人員多注意什麼？

> ✏️ 你的答案：


<details><summary>💡 卡住了再打開：參考答案</summary>

**參考答案：**第 81 位：`iadl` 只有 1 分（+0.13）、`adl_barthel` 只有 45 分（+0.10），加上中風、跌倒，把機率推到 0.80；但他的 SPMSQ 只錯 3 題（−0.04），認知其實沒有那麼差。模型把「日常生活功能差」當成失智的訊號，但功能差也可能來自中風、跌倒等身體因素。解釋可以提醒照顧人員：高風險不等於失智，要搭配認知評估，並轉介醫師。

</details>

## 2.2 依賴圖（scatter）：特徵值怎麼影響預測？
蜂群圖只告訴我們「紅點偏右」，看不出**曲線的形狀**。依賴圖把**一個特徵**畫成：
- **橫軸**：這個特徵的值（例如睡眠時數）
- **縱軸**：這個特徵的 SHAP 值（對失智機率的影響）
- **每個點是一位住民**

先看 SPMSQ：

In [ ]:
shap.plots.scatter(sv1[:, "spmsq_errors"], show=False)
plt.title("SPMSQ 錯誤題數 → 對失智機率的影響")
plt.show()

SPMSQ 錯 0–3 題的 SHAP 值是**負的**（拉低風險），錯 4 題以上變成**正的**（推高風險），分界在 3–4 題，和 SPMSQ 量表本身的判讀（錯 3 題以上為認知功能障礙）很接近。

### 🔓 揭曉：睡眠之謎
W7 留下的問題：W6 看到睡眠和失智是 **U 型**，羅吉斯回歸只能畫直線。隨機森林怎麼看睡眠？

In [ ]:
shap.plots.scatter(sv1[:, "sleep_hours"], show=False)
plt.axhline(0, color="gray", linestyle="--")
plt.title("隨機森林：睡眠時數 → 對失智機率的影響")
plt.xlabel("睡眠時數（小時）")
plt.show()

睡 **6–8 小時**的住民，SHAP 值是負的（拉低風險）；睡**太少**（< 5）或**太多**（> 9）的住民，SHAP 值是正的（推高風險）：**U 型**，隨機森林抓到了。

再看身高。身高和失智完全無關，SHAP 怎麼說？

In [ ]:
shap.plots.scatter(sv1[:, "height_cm"], show=False)
plt.axhline(0, color="gray", linestyle="--")
plt.title("隨機森林：身高 → 對失智機率的影響")
plt.xlabel("身高（公分）")
plt.show()

所有的點都貼著 0 附近，沒有任何形狀：**身高對預測幾乎沒有影響**，和我們的預期一致（模擬資料中身高和失智無關）。

### 羅吉斯回歸的睡眠呢？
羅吉斯回歸是線性模型，用 `shap.LinearExplainer`。因為我們把標準化放在 pipeline 裡，要先把資料標準化，再交給解釋器（單位是 log-odds，不是機率，**數字不能直接和隨機森林比**，但圖的形狀可以比）：

In [ ]:
scaler, lr = logit[0], logit[-1]                        # pipeline 的第一步（標準化）與最後一步（羅吉斯回歸）
Xtr_s = pd.DataFrame(scaler.transform(X_train), columns=X.columns)
Xte_s = pd.DataFrame(scaler.transform(X_test), columns=X.columns)

lin = shap.LinearExplainer(lr, shap.maskers.Independent(Xtr_s, max_samples=len(Xtr_s)))
lsv = lin(Xte_s)
lsv.data = X_test.values                                 # 圖的橫軸改回原本的單位（小時），不是標準化後的數字

shap.plots.scatter(lsv[:, "sleep_hours"], show=False)
plt.axhline(0, color="gray", linestyle="--")
plt.title("羅吉斯回歸：睡眠時數 → 對失智機率的影響")
plt.xlabel("睡眠時數（小時）")
plt.show()

羅吉斯回歸的點全部排在**一條直線**上：睡越多越危險（或睡越少越危險），**看不到 U 型**。這就是 W7 說的：線性模型抓不到非線性的關係。

> 💡 SHAP 不只解釋模型，也能**比較兩個模型「學到了什麼」**。兩個模型正確率差不多，但學到的睡眠規則完全不同。

---
## ▶ 實作 9-3（約 12 分鐘）畫依賴圖
> 🆘 **卡住了怎麼辦**：① 看錯誤訊息的最後一行 → ② 問 Colab 的 Gemini（按「解釋錯誤」，或請它用中文解釋）→ ③ 還是卡住，再打開下面的「💡 卡住了再打開」。看懂之後，自己再打一次。

**(a)** 畫出隨機森林 `age`（年齡）的依賴圖。年齡越大，SHAP 值怎麼變化？

In [ ]:
# 範例：shap.plots.scatter(sv1[:, "sleep_hours"], show=False)
# TODO：改成 age，加上標題 plt.title("...")
plt.show()


In [ ]:
#@title 💡 卡住了再打開：參考解答 { display-mode: "form" }
shap.plots.scatter(sv1[:, "age"], show=False)
plt.title("隨機森林：年齡 → 對失智機率的影響")
plt.show()

> ✏️ 你的答案：


<details><summary>💡 卡住了再打開：參考答案</summary>

**參考答案：**年齡越大，SHAP 值越高：75 歲以下是負的（拉低風險），約 80–85 歲之後才變成正的（推高風險）。

</details>

**(b)** 用 `color=` 參數，讓睡眠時數的依賴圖用**年齡**上色：`shap.plots.scatter(sv1[:, "sleep_hours"], color=sv1[:, "age"])`。圖上同樣睡 9 小時以上的住民，顏色不同的點（年齡不同），SHAP 值有差別嗎？

In [ ]:
# TODO：照題目的範例，畫出睡眠時數的依賴圖並用年齡上色
plt.show()


In [ ]:
#@title 💡 卡住了再打開：參考解答 { display-mode: "form" }
shap.plots.scatter(sv1[:, "sleep_hours"], color=sv1[:, "age"], show=False)
plt.title("睡眠時數的影響，依年齡上色")
plt.show()

> ✏️ 你的答案：


<details><summary>💡 卡住了再打開：參考答案</summary>

**參考答案：**（請看自己畫出的圖作答）睡很多的住民裡，顏色（年齡）混在一起，沒有很明顯的分層，表示睡眠的影響大致獨立於年齡；如果顏色分層明顯，表示有「交互作用」，兩個特徵會互相影響對方的作用。

</details>

**(c)** 比較隨機森林與羅吉斯回歸：`iadl`（工具性日常生活活動）的依賴圖，形狀一樣嗎？（提示：隨機森林用 `sv1[:, "iadl"]`，羅吉斯回歸用 `lsv[:, "iadl"]`，各畫一張）

In [ ]:
# TODO：畫出兩張圖（隨機森林一張、羅吉斯回歸一張），各加上標題


In [ ]:
#@title 💡 卡住了再打開：參考解答 { display-mode: "form" }
shap.plots.scatter(sv1[:, "iadl"], show=False)
plt.title("隨機森林：IADL")
plt.show()

shap.plots.scatter(lsv[:, "iadl"], show=False)
plt.title("羅吉斯回歸：IADL")
plt.show()
# 隨機森林是階梯狀（IADL 分數很低或很高才有明顯影響）；羅吉斯回歸是一條直線

> ✏️ 你的答案：


<details><summary>💡 卡住了再打開：參考答案</summary>

**參考答案：**羅吉斯回歸是一條**直線**：IADL 每增加一分，風險按固定幅度下降；隨機森林是**階梯狀**：IADL 低於約 4 分是正的（推高風險），4 分以上是負的（拉低風險），4 分以上之後幾乎不再變化。兩者方向相同（IADL 越高風險越低），形狀不同。

</details>

---
# 第 3 堂｜SHAP 的限制與負責任的使用
## 3.1 SHAP 解釋的是「模型」，不是「世界」
> **SHAP 說 SPMSQ 錯誤越多越危險，不等於「SPMSQ 錯誤造成失智」。** 它只告訴我們：這個模型，是怎麼用 SPMSQ 做預測的。

使用 SHAP 時要注意四件事：

| 注意事項 | 說明 |
|---|---|
| **1. 相關 ≠ 因果** | SHAP 反映的是資料裡的**相關**。如果「睡眠太多」只是失智的**症狀**（不是原因），SHAP 一樣會說睡眠很重要。不能因此說「改善睡眠可以預防失智」。 |
| **2. 相關的特徵會互相分攤** | 兩個特徵講的是同一件事（例如體重和 BMI），SHAP 會把功勞**分成兩份**，每一份都變小。下面實驗。 |
| **3. 模型錯，解釋也是錯的** | SHAP 解釋的是這個模型。模型本身不準，或訓練資料有偏誤（例如只有某幾家機構的住民），解釋就不能信。 |
| **4. 資料只有「預測當下」拿得到的資訊** | 確診之後才有的資料（CDR 分期、失智症用藥）一放進去，SHAP 會顯示它是最重要的特徵：這時**結果是漂亮的、但完全不能用**。看到可疑的第一名，要回頭檢查特徵。 |

### 實驗：相關的特徵會分攤功勞
把 `spmsq_errors` 複製一份叫 `spmsq_copy`（兩個特徵**完全一樣**），重新訓練隨機森林，看看 SHAP 怎麼分配：

In [ ]:
X2 = X.copy()
X2["spmsq_copy"] = X2["spmsq_errors"]            # 完全相同的第二份
X2_train, X2_test, _, _ = train_test_split(X2, y, test_size=0.25, stratify=y, random_state=1151)

forest2 = RandomForestClassifier(random_state=1151).fit(X2_train, y_train)
sv2 = shap.TreeExplainer(forest2)(X2_test)[:, :, 1]

two = pd.Series(np.abs(sv2.values).mean(axis=0), index=X2.columns).sort_values(ascending=False)
print("加了複製品之後（平均 |SHAP|）：")
print(two.head(4).round(3))
print("\n原本 spmsq_errors 的平均 |SHAP|：", round(shap_imp["spmsq_errors"], 3))

原本 SPMSQ 的平均 |SHAP| 是 0.163；多一份複製品之後，兩個特徵**合起來**大約還是 0.2，但各自變成 0.11 和 0.09。**如果只看單一特徵的數字，會以為 SPMSQ 沒那麼重要了。**
所以看到兩個意思相近的特徵時（體重、BMI、身高），要把它們合起來看，或只留一個。

### 想一想（不用寫程式）
**情境 A**：你把蜂群圖給機構主任看，主任說：「所以只要讓住民睡 7 小時，就可以預防失智？」你怎麼回答？

**情境 B**：同學的模型在蜂群圖上，最上面的特徵是「是否在服用失智症藥物」，正確率 99%。你覺得這個模型可以用在入住時的篩檢嗎？為什麼？

> ✏️ 你的答案：


<details><summary>💡 卡住了再打開：參考答案</summary>

**參考答案：**
> A：不能這樣說。SHAP 顯示的是**相關**：睡太少或太多的住民，模型預測的失智機率比較高，但不知道是睡眠造成失智、失智造成睡眠改變，還是另有原因。要知道「改變睡眠會不會影響失智」，需要做**實驗研究**（例如隨機分派）。
> B：不能。「正在服用失智症藥物」是**確診之後**才有的資訊，入住篩檢時拿不到；用它預測等於偷看答案，正確率再高也沒有意義。要拿掉這個特徵，用入住當下拿得到的評估資料重新訓練。

</details>

## 3.2 怎麼把 SHAP 用在機構？
- **向主任／家屬說明**：用瀑布圖的前 3 名，轉成白話：「這位阿嬤被列為高風險，主要是因為：SPMSQ 答錯 8 題、日常生活活動量表分數偏低、過去一年跌倒 2 次。」
- **檢查模型有沒有學到奇怪的東西**：蜂群圖裡出現「血型」、「住在哪一家機構」排前面，就要懷疑資料有偏誤。
- **決定要補什麼資料**：模型很依賴 SPMSQ，就要確保每位住民都有做 SPMSQ。
- **永遠由人做最後決定**：SHAP 是輔助，不是診斷。

---
## ▶ 實作 9-4（約 15 分鐘）向主任報告
> 🆘 **卡住了怎麼辦**：① 看錯誤訊息的最後一行 → ② 問 Colab 的 Gemini（按「解釋錯誤」，或請它用中文解釋）→ ③ 還是卡住，再打開下面的「💡 卡住了再打開」。看懂之後，自己再打一次。

請用 **3–4 句話**回答主任一開始的兩個問題（用日常用語，不要用「SHAP 值」「特徵」這類術語）：
1. 整體來說，模型最看重哪些資料？（根據 1.4 的蜂群圖，講出前 3 名和方向）
2. 剛才那位預測機率最高的住民（`i`），為什麼被判成高風險？

最後加上一句**提醒**：這個結果有什麼限制？

> ✏️ 你的答案：


<details><summary>💡 卡住了再打開：參考答案</summary>

**參考答案：**整體來說，模型最看重三項：認知測驗（SPMSQ）答錯越多、日常生活功能（巴氏量表、IADL）分數越低，被判成高風險的機會越高。第 82 位住民被判成高風險（預測機率 0.99），主要是因為認知測驗答錯 6 題、睡眠時間長達 9.8 小時、日常生活功能（IADL）只有 1 分。提醒：這是模型根據模擬資料整理出的「相關」，不代表原因，而且高風險不等於確診，仍要轉介醫師評估。

</details>

### 🏆 挑戰：用 SHAP 的發現改善模型
SHAP 發現睡眠是 U 型，羅吉斯回歸抓不到。W7 提過的辦法：自己加上一個特徵「**離 7 小時多遠**」。請：
1. 新增欄位 `sleep_dev = (sleep_hours - 7).abs()`
2. 重新切分、訓練羅吉斯回歸（參考上面「回顧」那一格）
3. 比較加入前後的**測試集正確率**，再用 `LinearExplainer` 看 `sleep_dev` 的依賴圖是不是一條「往右上」的直線

In [ ]:
# 參考：上面「回顧」那一格的做法
# TODO：Xd = X.copy()，新增 Xd["sleep_dev"]，切分、訓練，比較正確率（選做：畫 sleep_dev 的依賴圖）


In [ ]:
#@title 💡 卡住了再打開：參考解答 { display-mode: "form" }
Xd = X.copy()
Xd["sleep_dev"] = (Xd["sleep_hours"] - 7).abs()           # 離 7 小時多遠
Xd_train, Xd_test, _, _ = train_test_split(Xd, y, test_size=0.25, stratify=y, random_state=1151)

logit2 = make_pipeline(StandardScaler(), LogisticRegression()).fit(Xd_train, y_train)
print("原本的羅吉斯回歸：", round(logit.score(X_test, y_test), 3))
print("加入 sleep_dev：   ", round(logit2.score(Xd_test, y_test), 3))

sc2, lr2 = logit2[0], logit2[-1]
Xd_tr_s = pd.DataFrame(sc2.transform(Xd_train), columns=Xd.columns)
Xd_te_s = pd.DataFrame(sc2.transform(Xd_test), columns=Xd.columns)
lsv2 = shap.LinearExplainer(lr2, shap.maskers.Independent(Xd_tr_s, max_samples=len(Xd_tr_s)))(Xd_te_s)
lsv2.data = Xd_test.values

shap.plots.scatter(lsv2[:, "sleep_dev"], show=False)
plt.title("羅吉斯回歸：離 7 小時越遠 → 風險越高")
plt.xlabel("睡眠時數離 7 小時的距離（小時）")
plt.show()
# 正確率從 0.841 提高到 0.862：SHAP 發現的 U 型，變成一個新特徵，線性模型也學得會

---
### 本週 SHAP 指令總整理
| 目的 | 指令 |
|---|---|
| 安裝與版本 | `%pip install -q shap==0.52.0`、`shap.__version__` |
| 建立解釋器（樹模型） | `explainer = shap.TreeExplainer(forest)` |
| 算 SHAP 值 | `sv = explainer(X_test)`；`sv1 = sv[:, :, 1]`（取「失智」那一類） |
| 整體：重要性 | `shap.plots.bar(sv1, max_display=10)` |
| 整體：方向與大小 | `shap.plots.beeswarm(sv1, max_display=10)` |
| 單一住民 | `shap.plots.waterfall(sv1[i])`（`i` ＝ 第幾位） |
| 單一特徵的形狀 | `shap.plots.scatter(sv1[:, "sleep_hours"], color=sv1[:, "age"])` |
| 線性模型 | `shap.LinearExplainer(模型, shap.maskers.Independent(標準化後的訓練資料))` |
| 加標題（圖都要加 `show=False`） | 畫圖時加 `show=False`，再用 `plt.title("...")`、`plt.show()` |
| 驗證 | 基準值 `sv1.base_values[i]` ＋ `sv1.values[i].sum()` ＝ `predict_proba` |
| 查手冊 | <https://shap.readthedocs.io/>、`函數名稱?` |